In [1]:
import os
import numpy as np
import pandas as pd

N_CLASSES = 8
VAL_N = 200            # per class  (must match subsample.ipynb)
TEST_N = 200           # per class  (must match subsample.ipynb)
HOLDOUT_SEED = 0       # same dedicated holdout seed -> same val/test rows
MC_SEEDS = [1, 2, 3, 4, 5]

# TimeGAN scenes: contiguous blocks. Scene 0 (69567) is intentionally EXCLUDED —
# a contiguous block that large cannot avoid the scattered val/test rows
# (F3M only has ~70k rows in total).
SCENE_TRAIN = {1: 1000, 2: 500, 3: 100, 4: 50, 5: 30, 6: 10}

RAW_DIR = "CSV_Files"          # where F{i}M.csv live (same as subsample.ipynb)
OUT_DIR = "TimeGANs_csv"       # TimeGAN-specific scene tree

SEQ_LEN = 24                   # planned TimeGAN window length (feasibility report only)


# Val/Test reference (firewall)
Reproduces the **same** deterministic holdout as `subsample.ipynb` (`HOLDOUT_SEED = 0`) to learn which ORIGINAL row positions are in val/test. We do **not** rewrite `eval/` — the fixed yardstick stays shared with the other GANs. We only need the holdout row mask so contiguous blocks can avoid those rows (no leakage).

In [2]:
holdout_mask = {}      # class -> bool array over ORIGINAL row order (True = in val/test)
raw = {}               # class -> raw dataframe (with Labels appended)

for i in range(N_CLASSES):
    df = pd.read_csv(f"{RAW_DIR}/F{i}M.csv")
    df["Labels"] = i
    raw[i] = df

    # identical shuffle to subsample.ipynb; take the ORIGINAL indices of the
    # first VAL_N + TEST_N shuffled rows -> those are the val/test rows
    shuffled_index = df.sample(frac=1.0, random_state=HOLDOUT_SEED).index
    m = np.zeros(len(df), dtype=bool)
    m[shuffled_index[:VAL_N + TEST_N]] = True
    holdout_mask[i] = m

print({i: int(holdout_mask[i].sum()) for i in range(N_CLASSES)}, "holdout rows per class")


{0: 400, 1: 400, 2: 400, 3: 400, 4: 400, 5: 400, 6: 400, 7: 400} holdout rows per class


# TimeGAN scenario train sets (contiguous)
One contiguous block per (class, seed); scenes are nested prefixes of the block (scene6 ⊂ scene5 ⊂ … ⊂ scene1), mirroring `subsample.ipynb`. Start offsets are chosen per seed from well-separated zones of the recording, preferring blocks with **zero** val/test rows; if a class has too few clean stretches (F3), the block drops its few holdout rows and extends to keep the row budget (reported in the manifest).

In [3]:
BIGGEST = max(SCENE_TRAIN.values())

def pick_block(mask, need, seed, n_zones=len(MC_SEEDS)):
    """Choose a contiguous block of `need` rows for this seed.
    Prefer starts whose window contains ZERO holdout rows; pick within the seed's
    zone of the recording so seeds are spread apart. Fallback: minimal-overlap
    start in the zone (its holdout rows are dropped later)."""
    n = len(mask)
    cs = np.concatenate([[0], np.cumsum(mask)])
    win = cs[need:] - cs[:-need]                  # holdout count per start
    zone = (seed - 1) % n_zones
    lo, hi = int(zone * (n - need) / n_zones), int((zone + 1) * (n - need) / n_zones)
    zwin = win[lo:hi]
    clean = np.where(zwin == 0)[0]
    rng = np.random.default_rng(seed * 100003 + 17)
    if len(clean):
        return lo + int(clean[rng.integers(len(clean))]), 0
    best = int(zwin.min())
    cand = np.where(zwin == best)[0]
    return lo + int(cand[rng.integers(len(cand))]), best

manifest = []
blocks = {}     # (class, seed) -> clean contiguous array (BIGGEST, 15) w/o holdout rows

for i in range(N_CLASSES):
    arr = raw[i].to_numpy()
    m = holdout_mask[i]
    for seed in MC_SEEDS:
        start, overlap = pick_block(m, BIGGEST, seed)
        # take a bit extra so that, after dropping holdout rows, BIGGEST remain
        end = start + BIGGEST
        while (end - start) - m[start:end].sum() < BIGGEST and end < len(arr):
            end += 1
        keep = ~m[start:end]
        blk = arr[start:end][keep][:BIGGEST]
        blocks[(i, seed)] = blk
        manifest.append({"class": i, "seed": seed, "start_row": start,
                         "end_row": end, "holdout_dropped": int(m[start:end].sum()),
                         "rows": len(blk),
                         "t_start": float(blk[0, 0]), "t_end": float(blk[-1, 0])})

cols = list(raw[0].columns)
for seed in MC_SEEDS:
    for scene, n_train in SCENE_TRAIN.items():
        train_frames = [pd.DataFrame(blocks[(i, seed)][:n_train], columns=cols)
                        for i in range(N_CLASSES)]
        df_train = pd.concat(train_frames, ignore_index=True)
        out_dir = f"{OUT_DIR}/seed{seed}/scene{scene}"
        os.makedirs(out_dir, exist_ok=True)
        df_train.to_csv(f"{out_dir}/train.csv", index=False)

pd.DataFrame(manifest).to_csv(f"{OUT_DIR}/block_manifest.csv", index=False)
mf = pd.DataFrame(manifest)
print("blocks with holdout rows dropped:", int((mf.holdout_dropped > 0).sum()),
      "| max dropped in one block:", int(mf.holdout_dropped.max()))


blocks with holdout rows dropped: 5 | max dropped in one block: 2


In [4]:
print("Done. Layout:")
print(f"  {OUT_DIR}/seed<1-5>/scene<1-6>/train.csv   (contiguous, TimeGAN-only)")
print(f"  {OUT_DIR}/block_manifest.csv               (provenance of every block)")
print(f"  eval/ val+test: UNCHANGED — shared fixed yardstick, zero leakage")
print()
print(f"window feasibility at SEQ_LEN={SEQ_LEN} (stride-1):")
for k, n in SCENE_TRAIN.items():
    w = max(0, n - SEQ_LEN + 1)
    note = "" if w else "   <-- no full window: TimeGAN not evaluable"
    print(f"  scene {k}: {n:>4} rows/class -> {w:>4} windows/class{note}")


Done. Layout:
  TimeGANs_csv/seed<1-5>/scene<1-6>/train.csv   (contiguous, TimeGAN-only)
  TimeGANs_csv/block_manifest.csv               (provenance of every block)
  eval/ val+test: UNCHANGED — shared fixed yardstick, zero leakage

window feasibility at SEQ_LEN=24 (stride-1):
  scene 1: 1000 rows/class ->  977 windows/class
  scene 2:  500 rows/class ->  477 windows/class
  scene 3:  100 rows/class ->   77 windows/class
  scene 4:   50 rows/class ->   27 windows/class
  scene 5:   30 rows/class ->    7 windows/class
  scene 6:   10 rows/class ->    0 windows/class   <-- no full window: TimeGAN not evaluable
